In [ ]:
#1
!pip -q install transformers scikit-learn pandas openpyxl

import random, numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
#2
from google.colab import files
files.upload()   # اختاري Whisper_Small_Baseline_Results.xlsx

df = pd.read_excel("Whisper_Small_Baseline_Results.xlsx")
df["text"] = df["whisper_prediction"].astype(str).str.strip()
df["intent"] = df["intent"].str.strip().str.lower()   # نوحّد الأسماء (Billing -> billing)
df["escalation"] = df["escalation"].astype(int)
df = df[df["text"] != ""].reset_index(drop=True)

print(df.shape)
print(df["intent"].value_counts())
print(df["escalation"].value_counts())

Saving Whisper_Small_Baseline_Results.xlsx to Whisper_Small_Baseline_Results.xlsx
(80, 10)
intent
information     19
technical       14
cancellation    13
billing         12
complaint       12
out_of_scope    10
Name: count, dtype: int64
escalation
0    60
1    20
Name: count, dtype: int64


In [ ]:
#3
from transformers import AutoTokenizer

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lens = [len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in df["text"]]

print("Percentiles 50/90/95/100:", np.percentile(lens, [50, 90, 95, 100]))
print("Calls longer than 256:", sum(l > 256 for l in lens), "of", len(lens))
print("Calls longer than 512:", sum(l > 512 for l in lens), "of", len(lens))

Percentiles 50/90/95/100: [164.5  240.4  262.35 282.  ]
Calls longer than 256: 5 of 80
Calls longer than 512: 0 of 80


In [ ]:
#4
import re, warnings
from sklearn.model_selection import StratifiedKFold

MAX_LEN = 320   # كل المكالمات تدخل كاملة (أطولها 282)

label_names = ["billing", "information", "cancellation",
               "complaint", "technical", "out_of_scope"]
label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for l, i in label2id.items()}
df["intent_id"] = df["intent"].map(label2id)
assert df["intent_id"].notna().all()

# تطبيع عربي بسيط: تشكيل، تطويل، توحيد الألف والياء، الأرقام الهندية
def normalize_ar(t):
    t = re.sub(r"[\u064B-\u0652\u0640]", "", t)      # تشكيل + تطويل
    t = re.sub(r"[إأآ]", "ا", t)                      # توحيد الألف
    t = t.replace("ى", "ي")
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    return re.sub(r"\s+", " ", t).strip().lower()

df["text_norm"] = df["text"].apply(normalize_ar)

# 5-fold: نوزع حسب intent + escalation مع بعض
df["strata"] = df["intent"] + "_" + df["escalation"].astype(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # تحذير متوقع: billing_1 فيها عينتين بس
    folds = list(skf.split(df, df["strata"]))

for i, (tr, te) in enumerate(folds):
    print(f"Fold {i}: train={len(tr)} test={len(te)} "
          f"| escalation positives in test={df.loc[te, 'escalation'].sum()}")

Fold 0: train=64 test=16 | escalation positives in test=4
Fold 1: train=64 test=16 | escalation positives in test=4
Fold 2: train=64 test=16 | escalation positives in test=4
Fold 3: train=64 test=16 | escalation positives in test=3
Fold 4: train=64 test=16 | escalation positives in test=5


In [ ]:
#5
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

def make_tfidf():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=10)),
    ])

def cv_tfidf(text_col):
    intent_f1, esc_p, esc_r, esc_f1 = [], [], [], []
    oof_intent = np.zeros(len(df), dtype=int)
    for tr, te in folds:
        X_tr, X_te = df.loc[tr, text_col], df.loc[te, text_col]
        # intent
        m = make_tfidf().fit(X_tr, df.loc[tr, "intent_id"])
        p = m.predict(X_te); oof_intent[te] = p
        intent_f1.append(f1_score(df.loc[te, "intent_id"], p, average="macro"))
        # escalation
        m2 = make_tfidf().fit(X_tr, df.loc[tr, "escalation"])
        pe = m2.predict(X_te); y = df.loc[te, "escalation"]
        esc_p.append(precision_score(y, pe, zero_division=0))
        esc_r.append(recall_score(y, pe, zero_division=0))
        esc_f1.append(f1_score(y, pe, zero_division=0))
    ms = lambda a: f"{np.mean(a):.3f} ± {np.std(a):.3f}"
    print(f"[{text_col}] Intent Macro-F1: {ms(intent_f1)}")
    print(f"[{text_col}] Escalation  P: {ms(esc_p)} | R: {ms(esc_r)} | F1: {ms(esc_f1)}")
    return oof_intent

print("=== TF-IDF (5-fold CV) ===")
oof_raw  = cv_tfidf("text")        # بدون تطبيع
oof_norm = cv_tfidf("text_norm")   # مع التطبيع

print("\nPer-class report (normalized, كل الـ folds مجمّعة):")
print(classification_report(df["intent_id"], oof_norm,
                            target_names=label_names, digits=3, zero_division=0))

=== TF-IDF (5-fold CV) ===
[text] Intent Macro-F1: 0.952 ± 0.025
[text] Escalation  P: 0.910 ± 0.111 | R: 0.630 ± 0.312 | F1: 0.677 ± 0.182
[text_norm] Intent Macro-F1: 0.952 ± 0.025
[text_norm] Escalation  P: 0.843 ± 0.135 | R: 0.630 ± 0.312 | F1: 0.658 ± 0.187

Per-class report (normalized, كل الـ folds مجمّعة):
              precision    recall  f1-score   support

     billing      1.000     1.000     1.000        12
 information      0.950     1.000     0.974        19
cancellation      1.000     0.846     0.917        13
   complaint      0.846     0.917     0.880        12
   technical      0.929     0.929     0.929        14
out_of_scope      1.000     1.000     1.000        10

    accuracy                          0.950        80
   macro avg      0.954     0.949     0.950        80
weighted avg      0.953     0.950     0.950        80



In [ ]:
#6
import copy, time, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

PAD_ID = tokenizer.pad_token_id

class CallDS(Dataset):
    def __init__(self, texts, intents, escs):
        self.ids = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]
        self.intents, self.escs = list(intents), list(escs)
    def __len__(self): return len(self.intents)
    def __getitem__(self, i):
        return self.ids[i], self.intents[i], self.escs[i]

def collate(batch):
    L = max(len(b[0]) for b in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (x, _, _) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
    return (ids, mask,
            torch.tensor([b[1] for b in batch]),
            torch.tensor([b[2] for b in batch]))

class CallSenseModel(nn.Module):
    def __init__(self, name, n_intents, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(name)
        h = self.bert.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.intent_head = nn.Linear(h, n_intents)
        self.esc_head = nn.Linear(h, 2)
    def forward(self, ids, mask):
        out = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).float()
        pooled = self.drop((out * m).sum(1) / m.sum(1).clamp(min=1))   # mean pooling
        return self.intent_head(pooled), self.esc_head(pooled)

def get_probs(model, loader):
    model.eval(); pi, pe = [], []
    with torch.no_grad():
        for ids, mask, _, _ in loader:
            li, le = model(ids.to(device), mask.to(device))
            pi.append(torch.softmax(li, -1).cpu())
            pe.append(torch.softmax(le, -1)[:, 1].cpu())
    return torch.cat(pi).numpy(), torch.cat(pe).numpy()

def run_fold(k, epochs=20, lr=3e-5, head_lr=1e-3, bs=8, esc_w=2.0, verbose=True):
    torch.manual_seed(42 + k); np.random.seed(42 + k)
    tr_idx, te_idx = folds[k]
    tr_df, te_df = df.loc[tr_idx], df.loc[te_idx]
    fit_df, val_df = train_test_split(tr_df, test_size=0.2, random_state=k,
                                      stratify=tr_df["intent"])
    mk = lambda d, sh: DataLoader(CallDS(d["text"], d["intent_id"], d["escalation"]),
                                  batch_size=bs, shuffle=sh, collate_fn=collate)
    fit_loader, val_loader, te_loader = mk(fit_df, True), mk(val_df, False), mk(te_df, False)

    model = CallSenseModel(MODEL_NAME, len(label_names)).to(device)
    opt = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": lr},
        {"params": list(model.intent_head.parameters()) + list(model.esc_head.parameters()),
         "lr": head_lr}], weight_decay=0.01)
    total = epochs * len(fit_loader)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    ce_i = nn.CrossEntropyLoss()
    ce_e = nn.CrossEntropyLoss(weight=torch.tensor([1.0, esc_w], device=device))

    best_score, best_state, best_ep = -1, None, 0
    for ep in range(1, epochs + 1):
        model.train(); tl = 0
        for ids, mask, yi, ye in fit_loader:
            ids, mask, yi, ye = ids.to(device), mask.to(device), yi.to(device), ye.to(device)
            opt.zero_grad()
            li, le = model(ids, mask)
            loss = ce_i(li, yi) + ce_e(le, ye)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tl += loss.item()
        pi, pe = get_probs(model, val_loader)
        v_if1 = f1_score(val_df["intent_id"], pi.argmax(1), average="macro", zero_division=0)
        v_ef1 = f1_score(val_df["escalation"], (pe > 0.5).astype(int), zero_division=0)
        score = v_if1 + v_ef1
        if score > best_score:
            best_score, best_ep = score, ep
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"ep{ep:02d} loss={tl/len(fit_loader):.3f} | val intent F1={v_if1:.3f} "
                  f"esc F1={v_ef1:.3f}{'  <- best' if best_ep == ep else ''}")

    model.load_state_dict(best_state)
    ti, te_ = get_probs(model, te_loader)
    vi, ve = get_probs(model, val_loader)
    res = dict(k=k, best_ep=best_ep, te_idx=te_idx, test_intent_probs=ti, test_esc_probs=te_,
               val_y_esc=val_df["escalation"].values, val_esc_probs=ve)
    del model, opt; torch.cuda.empty_cache()
    return res

def report_fold(r):
    te = df.loc[r["te_idx"]]
    pe = (r["test_esc_probs"] > 0.5).astype(int)
    print(f"\nFold {r['k']} | best epoch = {r['best_ep']}")
    print(f"  Intent Macro-F1 = {f1_score(te['intent_id'], r['test_intent_probs'].argmax(1), average='macro', zero_division=0):.3f}")
    print(f"  Escalation P={precision_score(te['escalation'], pe, zero_division=0):.3f} "
          f"R={recall_score(te['escalation'], pe, zero_division=0):.3f} "
          f"F1={f1_score(te['escalation'], pe, zero_division=0):.3f}")

In [ ]:
#7
t0 = time.time()
r0 = run_fold(0)
report_fold(r0)
print(f"\nTime: {time.time()-t0:.0f}s")

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ep01 loss=2.570 | val intent F1=0.155 esc F1=0.000  <- best
ep02 loss=2.329 | val intent F1=0.062 esc F1=0.000
ep03 loss=1.894 | val intent F1=0.633 esc F1=0.000  <- best
ep04 loss=1.180 | val intent F1=0.451 esc F1=0.333  <- best
ep05 loss=0.676 | val intent F1=0.567 esc F1=0.400  <- best
ep06 loss=0.371 | val intent F1=0.778 esc F1=0.400  <- best
ep07 loss=0.112 | val intent F1=0.778 esc F1=0.667  <- best
ep08 loss=0.054 | val intent F1=0.856 esc F1=0.400
ep09 loss=0.018 | val intent F1=0.778 esc F1=0.667
ep10 loss=0.006 | val intent F1=0.778 esc F1=0.500
ep11 loss=0.005 | val intent F1=0.778 esc F1=0.400
ep12 loss=0.003 | val intent F1=0.764 esc F1=0.667
ep13 loss=0.003 | val intent F1=0.778 esc F1=0.500
ep14 loss=0.001 | val intent F1=0.778 esc F1=0.400
ep15 loss=0.001 | val intent F1=0.778 esc F1=0.400
ep16 loss=0.001 | val intent F1=0.778 esc F1=0.500
ep17 loss=0.001 | val intent F1=0.778 esc F1=0.500
ep18 loss=0.001 | val intent F1=0.778 esc F1=0.500
ep19 loss=0.001 | val intent

In [ ]:
#8
EPOCHS = 10
results = []
t0 = time.time()
for k in range(5):
    r = run_fold(k, epochs=EPOCHS, verbose=False)
    results.append(r)
    report_fold(r)
print(f"\nTotal time: {time.time()-t0:.0f}s")

pd.to_pickle(results, "cv_results.pkl")   # حفظ احتياطي لو انقطع الاتصال

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fold 0 | best epoch = 10
  Intent Macro-F1 = 0.861
  Escalation P=1.000 R=0.500 F1=0.667


[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fold 1 | best epoch = 9
  Intent Macro-F1 = 0.755
  Escalation P=0.667 R=0.500 F1=0.571


[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fold 2 | best epoch = 8
  Intent Macro-F1 = 0.822
  Escalation P=1.000 R=0.500 F1=0.667


[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fold 3 | best epoch = 7
  Intent Macro-F1 = 0.911
  Escalation P=0.600 R=1.000 F1=0.750


[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Fold 4 | best epoch = 8
  Intent Macro-F1 = 0.567
  Escalation P=1.000 R=0.800 F1=0.889

Total time: 156s


In [ ]:
#9
def bert_fold_metrics(r, thr=0.5):
    te = df.loc[r["te_idx"]]
    pe = (r["test_esc_probs"] > thr).astype(int)
    return (f1_score(te["intent_id"], r["test_intent_probs"].argmax(1), average="macro", zero_division=0),
            precision_score(te["escalation"], pe, zero_division=0),
            recall_score(te["escalation"], pe, zero_division=0),
            f1_score(te["escalation"], pe, zero_division=0))

def tfidf_fold_metrics(k):
    tr, te = folds[k]
    m = make_tfidf().fit(df.loc[tr, "text"], df.loc[tr, "intent_id"])
    fi = f1_score(df.loc[te, "intent_id"], m.predict(df.loc[te, "text"]), average="macro")
    m2 = make_tfidf().fit(df.loc[tr, "text"], df.loc[tr, "escalation"])
    pe = m2.predict(df.loc[te, "text"]); y = df.loc[te, "escalation"]
    return (fi, precision_score(y, pe, zero_division=0),
            recall_score(y, pe, zero_division=0), f1_score(y, pe, zero_division=0))

rows = []
for r in results:
    rows.append([r["k"], r["best_ep"], *bert_fold_metrics(r), *tfidf_fold_metrics(r["k"])])

cmp = pd.DataFrame(rows, columns=[
    "fold", "best_ep",
    "BERT_intent", "BERT_escP", "BERT_escR", "BERT_escF1",
    "TFIDF_intent", "TFIDF_escP", "TFIDF_escR", "TFIDF_escF1"]).round(3)

display(cmp)
print(cmp.drop(columns=["fold", "best_ep"]).agg(["mean", "std"]).round(3).T)

,fold,best_ep,BERT_intent,BERT_escP,BERT_escR,BERT_escF1,TFIDF_intent,TFIDF_escP,TFIDF_escR,TFIDF_escF1
0,0,10,0.861,1.000,0.5,0.667,0.933,1.00,0.50,0.667
1,1,9,0.755,0.667,0.5,0.571,0.933,0.80,1.00,0.889
2,2,8,0.822,1.000,0.5,0.667,0.948,1.00,0.25,0.400
3,3,7,0.911,0.600,1.0,0.750,1.000,0.75,1.00,0.857
4,4,8,0.567,1.000,0.8,0.889,0.943,1.00,0.40,0.571


               mean    std
BERT_intent   0.783  0.134
BERT_escP     0.853  0.202
BERT_escR     0.660  0.230
BERT_escF1    0.709  0.119
TFIDF_intent  0.951  0.028
TFIDF_escP    0.910  0.124
TFIDF_escR    0.630  0.349
TFIDF_escF1   0.677  0.203


In [ ]:
#10
from sklearn.metrics import accuracy_score, confusion_matrix

# ---- نجمّع تنبؤات BERT خارج التدريب (out-of-fold) ----
oof_bert_int = np.zeros(len(df), dtype=int)
oof_bert_esc_prob = np.zeros(len(df))
for r in results:
    oof_bert_int[r["te_idx"]] = r["test_intent_probs"].argmax(1)
    oof_bert_esc_prob[r["te_idx"]] = r["test_esc_probs"]

# ---- نجمّع تنبؤات TF-IDF على نفس الـ folds ----
oof_tf_int = np.zeros(len(df), dtype=int)
oof_tf_esc = np.zeros(len(df), dtype=int)
for tr, te in folds:
    oof_tf_int[te] = make_tfidf().fit(df.loc[tr, "text"], df.loc[tr, "intent_id"]).predict(df.loc[te, "text"])
    oof_tf_esc[te] = make_tfidf().fit(df.loc[tr, "text"], df.loc[tr, "escalation"]).predict(df.loc[te, "text"])

y_int, y_esc = df["intent_id"].values, df["escalation"].values
oof_bert_esc = (oof_bert_esc_prob > 0.5).astype(int)

# ---- جدول الملخص ----
def summary(name, pi, pe):
    return {
        "model": name,
        "intent_acc": accuracy_score(y_int, pi),
        "intent_macroF1": f1_score(y_int, pi, average="macro"),
        "intent_macroP": precision_score(y_int, pi, average="macro", zero_division=0),
        "intent_macroR": recall_score(y_int, pi, average="macro", zero_division=0),
        "esc_acc": accuracy_score(y_esc, pe),
        "esc_P": precision_score(y_esc, pe, zero_division=0),
        "esc_R": recall_score(y_esc, pe, zero_division=0),
        "esc_F1": f1_score(y_esc, pe, zero_division=0),
    }

display(pd.DataFrame([summary("TF-IDF", oof_tf_int, oof_tf_esc),
                      summary("CAMeLBERT", oof_bert_int, oof_bert_esc)]).round(3).set_index("model").T)

# ---- تفاصيل CAMeLBERT ----
print("\n=== CAMeLBERT: Intent per-class (80 مكالمة مجمّعة) ===")
print(classification_report(y_int, oof_bert_int, target_names=label_names, digits=3, zero_division=0))
print("Intent confusion matrix:")
display(pd.DataFrame(confusion_matrix(y_int, oof_bert_int, labels=range(6)),
                     index=label_names, columns=label_names))

print("\n=== CAMeLBERT: Escalation (threshold = 0.5) ===")
print(classification_report(y_esc, oof_bert_esc, target_names=["no_esc", "esc"], digits=3, zero_division=0))
print("Escalation confusion matrix (صفوف = الحقيقة، أعمدة = التنبؤ):")
print(confusion_matrix(y_esc, oof_bert_esc))

model,TF-IDF,CAMeLBERT
intent_acc,0.950,0.800
intent_macroF1,0.950,0.796
intent_macroP,0.954,0.813
intent_macroR,0.949,0.791
esc_acc,0.875,0.875
esc_P,0.857,0.812
esc_R,0.600,0.650
esc_F1,0.706,0.722



=== CAMeLBERT: Intent per-class (80 مكالمة مجمّعة) ===
              precision    recall  f1-score   support

     billing      0.875     0.583     0.700        12
 information      0.818     0.947     0.878        19
cancellation      0.600     0.692     0.643        13
   complaint      0.727     0.667     0.696        12
   technical      0.857     0.857     0.857        14
out_of_scope      1.000     1.000     1.000        10

    accuracy                          0.800        80
   macro avg      0.813     0.791     0.796        80
weighted avg      0.807     0.800     0.797        80

Intent confusion matrix:


,billing,information,cancellation,complaint,technical,out_of_scope
billing,7,2,3,0,0,0
information,1,18,0,0,0,0
cancellation,0,1,9,3,0,0
complaint,0,0,2,8,2,0
technical,0,1,1,0,12,0
out_of_scope,0,0,0,0,0,10



=== CAMeLBERT: Escalation (threshold = 0.5) ===
              precision    recall  f1-score   support

      no_esc      0.891     0.950     0.919        60
         esc      0.812     0.650     0.722        20

    accuracy                          0.875        80
   macro avg      0.852     0.800     0.821        80
weighted avg      0.871     0.875     0.870        80

Escalation confusion matrix (صفوف = الحقيقة، أعمدة = التنبؤ):
[[57  3]
 [ 7 13]]


In [ ]:
#11
from sklearn.metrics import average_precision_score, fbeta_score, confusion_matrix
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def pick_threshold(y, p, beta=2.0):
    grid = np.arange(0.05, 0.95, 0.025)
    scores = np.array([fbeta_score(y, (p >= t).astype(int), beta=beta, zero_division=0) for t in grid])
    best_ts = grid[np.isclose(scores, scores.max())]
    return float(np.median(best_ts))            # نختار وسط أفضل منطقة، أثبت من حافتها

oof_tf_prob = np.zeros(len(df))
oof_tf_thr_pred = np.zeros(len(df), dtype=int)
thresholds = []

for k, (tr, te) in enumerate(folds):
    X_tr, y_tr = df.loc[tr, "text"], df.loc[tr, "escalation"]
    # 1) احتمالات "خارج التدريب" داخل الـ train فقط
    inner = StratifiedKFold(5, shuffle=True, random_state=k)
    p_inner = cross_val_predict(make_tfidf(), X_tr, y_tr, cv=inner, method="predict_proba")[:, 1]
    # 2) نختار العتبة من الـ train فقط
    thr = pick_threshold(y_tr.values, p_inner)
    thresholds.append(thr)
    # 3) نطبقها على الـ test fold
    m = make_tfidf().fit(X_tr, y_tr)
    p_te = m.predict_proba(df.loc[te, "text"])[:, 1]
    oof_tf_prob[te] = p_te
    oof_tf_thr_pred[te] = (p_te >= thr).astype(int)

print("Thresholds chosen per fold:", np.round(thresholds, 3))

def esc_summary(name, pred, prob):
    return {"setting": name,
            "P": precision_score(y_esc, pred, zero_division=0),
            "R": recall_score(y_esc, pred, zero_division=0),
            "F1": f1_score(y_esc, pred, zero_division=0),
            "F2": fbeta_score(y_esc, pred, beta=2, zero_division=0),
            "PR-AUC": average_precision_score(y_esc, prob)}

tbl = pd.DataFrame([
    esc_summary("TF-IDF @0.5",   oof_tf_esc,        oof_tf_prob),
    esc_summary("TF-IDF tuned",  oof_tf_thr_pred,   oof_tf_prob),
    esc_summary("BERT @0.5",     oof_bert_esc,      oof_bert_esc_prob),
]).round(3).set_index("setting")
display(tbl)

print("\nTF-IDF tuned: confusion matrix (صفوف = الحقيقة، أعمدة = التنبؤ)")
print(confusion_matrix(y_esc, oof_tf_thr_pred))

Thresholds chosen per fold: [0.4   0.425 0.375 0.4   0.275]


,P,R,F1,F2,PR-AUC
setting,,,,,
TF-IDF @0.5,0.857,0.60,0.706,0.638,0.805
TF-IDF tuned,0.783,0.90,0.837,0.874,0.805
BERT @0.5,0.812,0.65,0.722,0.677,0.817



TF-IDF tuned: confusion matrix (صفوف = الحقيقة، أعمدة = التنبؤ)
[[55  5]
 [ 2 18]]


In [ ]:
#12
avg_prob = (oof_tf_prob + oof_bert_esc_prob) / 2

def top_k_hits(p, k=20):
    return int(y_esc[np.argsort(-p)[:k]].sum())   # كم تصعيد حقيقي ضمن أعلى 20 درجة

cmp2 = pd.DataFrame({
    "PR-AUC": [average_precision_score(y_esc, oof_tf_prob),
               average_precision_score(y_esc, oof_bert_esc_prob),
               average_precision_score(y_esc, avg_prob)],
    "true esc in top-20": [top_k_hits(oof_tf_prob), top_k_hits(oof_bert_esc_prob), top_k_hits(avg_prob)],
}, index=["TF-IDF", "BERT", "Average"]).round(3)
display(cmp2)

,PR-AUC,true esc in top-20
TF-IDF,0.805,17
BERT,0.817,16
Average,0.771,16


In [ ]:
#13
import joblib, sklearn
from sklearn.model_selection import cross_val_predict, StratifiedKFold

# 1) تدريب نهائي على كل الداتا (نص خام، بدون تطبيع)
final_intent = make_tfidf().fit(df["text"], df["intent_id"])
final_esc    = make_tfidf().fit(df["text"], df["escalation"])

# 2) العتبة النهائية من احتمالات خارج التدريب على الـ 80
p_oof = cross_val_predict(make_tfidf(), df["text"], df["escalation"],
                          cv=StratifiedKFold(5, shuffle=True, random_state=42),
                          method="predict_proba")[:, 1]
ESC_THRESHOLD = pick_threshold(df["escalation"].values, p_oof)
print("Final escalation threshold:", ESC_THRESHOLD)

# 3) حفظ كل شي في ملف واحد
bundle = {
    "intent_model": final_intent,
    "escalation_model": final_esc,
    "label_names": label_names,
    "escalation_threshold": ESC_THRESHOLD,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, "callsense_classifier.joblib")
print("Saved. sklearn version:", sklearn.__version__)

# 4) دالة predict
def predict(transcript: str, bundle=bundle):
    text = (transcript or "").strip()
    if not text:
        return {"error": "empty transcript"}
    probs = bundle["intent_model"].predict_proba([text])[0]
    top = int(probs.argmax())
    esc_p = float(bundle["escalation_model"].predict_proba([text])[0][1])
    return {
        "intent": bundle["label_names"][top],
        "intent_confidence": round(float(probs[top]), 3),
        "intent_probs": {bundle["label_names"][i]: round(float(p), 3) for i, p in enumerate(probs)},
        "escalation": int(esc_p >= bundle["escalation_threshold"]),
        "escalation_probability": round(esc_p, 3),
        "escalation_threshold": bundle["escalation_threshold"],
    }

# 5) تجربة على نص جديد
sample = "السلام عليكم النت اندي فسل كلشوي وهذ ثالثيوم قل مرا تجولن بنحلها ومنحله وإذ مانهلت اليوم بلغي الشترك"
print(predict(sample))

from google.colab import files
files.download("callsense_classifier.joblib")

Final escalation threshold: 0.4000000000000001
Saved. sklearn version: 1.6.1
{'intent': 'cancellation', 'intent_confidence': 0.22, 'intent_probs': {'billing': 0.149, 'information': 0.194, 'cancellation': 0.22, 'complaint': 0.148, 'technical': 0.193, 'out_of_scope': 0.096}, 'escalation': 0, 'escalation_probability': 0.273, 'escalation_threshold': 0.4000000000000001}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#14
import os, json, joblib, sklearn, datetime, shutil, textwrap
import numpy as np
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             fbeta_score, average_precision_score)

PKG = "callsense_classifier_pkg"
MODEL_VERSION = "tfidf-lr-v1"
os.makedirs(PKG, exist_ok=True)
r4 = lambda x: round(float(x), 4)

# ---------- 1) المودل ----------
bundle = {
    "intent_model": final_intent,
    "escalation_model": final_esc,
    "label_names": label_names,
    "escalation_threshold": float(ESC_THRESHOLD),
    "model_version": MODEL_VERSION,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, f"{PKG}/callsense_classifier.joblib")

# ---------- 2) بطاقة المودل: البارامترز + النتايج ----------
def esc_metrics(pred, prob):
    return {"precision": r4(precision_score(y_esc, pred, zero_division=0)),
            "recall": r4(recall_score(y_esc, pred, zero_division=0)),
            "f1": r4(f1_score(y_esc, pred, zero_division=0)),
            "f2": r4(fbeta_score(y_esc, pred, beta=2, zero_division=0)),
            "pr_auc": r4(average_precision_score(y_esc, prob)),
            "missed_escalations": int(((y_esc == 1) & (pred == 0)).sum()),
            "false_alarms": int(((y_esc == 0) & (pred == 1)).sum())}

def intent_metrics(pred):
    return {"accuracy": r4(accuracy_score(y_int, pred)),
            "macro_f1": r4(f1_score(y_int, pred, average="macro")),
            "per_class_f1": {n: r4(v) for n, v in
                             zip(label_names, f1_score(y_int, pred, average=None, labels=range(6)))}}

model_card = {
    "model_version": MODEL_VERSION,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "task": "Arabic call-center transcript -> intent (6 classes) + escalation (0/1)",
    "input": "Whisper-small transcript text, raw (no normalization)",
    "labels": label_names,
    "training_data": {"n_calls": int(len(df)),
                      "intent_counts": df["intent"].value_counts().to_dict(),
                      "escalation_counts": {str(k): int(v) for k, v in df["escalation"].value_counts().items()},
                      "source": "team-recorded scripted mock calls, Whisper-small transcripts"},
    "parameters": {
        "vectorizer": {"type": "TfidfVectorizer", "analyzer": "char_wb", "ngram_range": [2, 5], "sublinear_tf": True},
        "classifier": {"type": "LogisticRegression", "C": 10, "class_weight": "balanced", "max_iter": 3000},
        "arabic_normalization": False,
        "escalation_threshold": float(ESC_THRESHOLD),
        "threshold_selection": "maximize F2 on out-of-fold probabilities (5-fold) of the training data",
    },
    "evaluation": {
        "protocol": "5-fold stratified CV (seed 42) on 80 calls; pooled out-of-fold predictions; "
                    "escalation threshold tuned inside each training fold only",
        "tfidf_intent": intent_metrics(oof_tf_int),
        "tfidf_escalation_at_0.5": esc_metrics(oof_tf_esc, oof_tf_prob),
        "tfidf_escalation_tuned_threshold": esc_metrics(oof_tf_thr_pred, oof_tf_prob),
        "camelbert_intent": intent_metrics(oof_bert_int),
        "camelbert_escalation_at_0.5": esc_metrics(oof_bert_esc, oof_bert_esc_prob),
        "decision": "TF-IDF + LogisticRegression selected: beat CAMeLBERT on intent in all 5 folds; "
                    "escalation PR-AUC similar (0.805 vs 0.817); averaging both did not help.",
    },
    "limitations": [
        "Only 80 scripted calls; 20 escalation positives -> small-sample estimates.",
        "Trained on Whisper-small transcripts (WER ~0.44); other ASR output may behave differently.",
        "Escalation threshold tuned for recall: expect more false alarms than misses.",
        "Escalation flag is decision support, not a replacement for human review.",
    ],
    "environment": {"scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
with open(f"{PKG}/model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, ensure_ascii=False, indent=2)

# ---------- 3) كود الاستخدام ----------
inference_code = '''
import os, json, datetime
import joblib

_HERE = os.path.dirname(os.path.abspath(__file__))


class CallSenseClassifier:
    """Takes a call transcript, returns intent + escalation flag."""

    def __init__(self, bundle_path=None, low_confidence_below=0.40):
        path = bundle_path or os.path.join(_HERE, "callsense_classifier.joblib")
        b = joblib.load(path)
        self.intent_model = b["intent_model"]
        self.escalation_model = b["escalation_model"]
        self.labels = b["label_names"]
        self.threshold = b["escalation_threshold"]
        self.version = b["model_version"]
        self.low_conf = low_confidence_below

    def predict(self, transcript, call_id=None):
        text = (transcript or "").strip()
        if not text:
            return {"call_id": call_id, "error": "empty transcript"}
        probs = self.intent_model.predict_proba([text])[0]
        top = int(probs.argmax())
        esc_p = float(self.escalation_model.predict_proba([text])[0][1])
        return {
            "call_id": call_id,
            "transcript": text,
            "intent": self.labels[top],
            "intent_confidence": round(float(probs[top]), 3),
            "intent_probs": {self.labels[i]: round(float(p), 3) for i, p in enumerate(probs)},
            "escalation": int(esc_p >= self.threshold),
            "escalation_probability": round(esc_p, 3),
            "escalation_threshold": self.threshold,
            "low_confidence": bool(float(probs[top]) < self.low_conf),
            "model_version": self.version,
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        }

    def predict_and_save(self, transcript, out_path="classified_calls.jsonl", call_id=None):
        """Predict, then append the result as one JSON line (handoff to the next model)."""
        result = self.predict(transcript, call_id=call_id)
        with open(out_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\\n")
        return result
'''
with open(f"{PKG}/callsense_inference.py", "w", encoding="utf-8") as f:
    f.write(inference_code.lstrip())

# ---------- 4) requirements + README ----------
with open(f"{PKG}/requirements.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\nnumpy=={np.__version__}\njoblib=={joblib.__version__}\n")

readme = f"""# CallSense classifier ({MODEL_VERSION})

Input: transcript text (Whisper output, as-is). Output: intent (6 classes) + escalation (0/1).

pip install -r requirements.txt

from callsense_inference import CallSenseClassifier
clf = CallSenseClassifier()
result = clf.predict(transcript, call_id="CS081")
clf.predict_and_save(transcript, "classified_calls.jsonl", call_id="CS081")  # one JSON line per call

Result keys: call_id, transcript, intent, intent_confidence, intent_probs, escalation,
escalation_probability, escalation_threshold, low_confidence, model_version, timestamp.
Details, parameters and evaluation: model_card.json
"""
with open(f"{PKG}/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print(os.listdir(PKG))

['requirements.txt', 'model_card.json', 'callsense_inference.py', 'README.md', 'callsense_classifier.joblib']


In [ ]:
#15
import sys, importlib
sys.path.insert(0, PKG)
import callsense_inference as cc
importlib.reload(cc)

clf = cc.CallSenseClassifier()

tests = {
    "T1": "السلام عليكم، النت عندي يفصل كل شوي وهذا ثالث يوم، كل مرة تقولون بنحلها وما انحلت، وإذا ما انحلت اليوم بلغي الاشتراك",
    "T2": "السلام عليكم، وش الباقات المتوفرة عندكم وكم سعر الباقة المفتوحة؟",
    "T3": "الفاتورة هذا الشهر جتني أعلى من المعتاد، ممكن توضح لي ليش؟",
    "T4": "اهلن انا قاق علكم ابغا اعرف كا سعر السيارة المعروده في الموكع حككم",
}
if os.path.exists("classified_calls.jsonl"):
    os.remove("classified_calls.jsonl")
for cid, txt in tests.items():
    res = clf.predict_and_save(txt, "classified_calls.jsonl", call_id=cid)
    print(cid, "->", res["intent"], f"({res['intent_confidence']})",
          "| escalation:", res["escalation"], f"(p={res['escalation_probability']})",
          "| low_conf:", res["low_confidence"])

print("\nسطور الملف المحفوظ:")
print(open("classified_calls.jsonl", encoding="utf-8").read()[:600])

# نسخة احتياطية في Drive + ملف zip للتحميل
from google.colab import drive, files
drive.mount("/content/drive")
shutil.copytree(PKG, "/content/drive/MyDrive/CallSense/" + PKG, dirs_exist_ok=True)
shutil.make_archive(PKG, "zip", PKG)
files.download(PKG + ".zip")

T1 -> technical (0.235) | escalation: 0 (p=0.303) | low_conf: True
T2 -> information (0.498) | escalation: 0 (p=0.164) | low_conf: False
T3 -> billing (0.411) | escalation: 0 (p=0.206) | low_conf: False
T4 -> information (0.228) | escalation: 0 (p=0.234) | low_conf: True

سطور الملف المحفوظ:
{"call_id": "T1", "transcript": "السلام عليكم، النت عندي يفصل كل شوي وهذا ثالث يوم، كل مرة تقولون بنحلها وما انحلت، وإذا ما انحلت اليوم بلغي الاشتراك", "intent": "technical", "intent_confidence": 0.235, "intent_probs": {"billing": 0.177, "information": 0.161, "cancellation": 0.225, "complaint": 0.114, "technical": 0.235, "out_of_scope": 0.088}, "escalation": 0, "escalation_probability": 0.303, "escalation_threshold": 0.4000000000000001, "low_confidence": true, "model_version": "tfidf-lr-v1", "timestamp": "2026-10-25T18:29:24"}
{"call_id": "T2", "transcript": "السلام عليكم، وش الباقات المتوفرة 
Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

full_tests = {
    "T1": "السلام عليكم مركز خدمة العمل كيف أقدر أساعدك؟ وعليكم السلام الانترنت عندي يفصل كل شوي وهذا ثالث يوم وانا اتصلت قبل مرتين ومحد حلها تفضل أستاذ ممكن رقم الجوال المسجل؟ صفر خمسة ثمانية سبعة أتحقق من الشبكه في منطقتك الحين طيب كل مرة تقولون بنحلها وما انحلت أنا تعبت صراحه ولو ما انحلت اليوم بلغي الاشتراك وأروح لشركه ثانيه أنا اسف على الازعاج راح أرفع بلاغ عاجل للفريق التقني ومشرفي بيتواصل معك هذا اليوم",
    "T2": "أهلا بك في خدمة العملية كيف أقدر ساعدك؟ السلام عليكم ابغى اعرف وش الباقات المتوفره عندكم وكم سعر الباقه المفتوحه تفضل عندنا باقات شهريه بأسعار مختلفه والباقه المفتوحه سعرها مية وتسعه وعشرين ريال شهريا طيب وهل أقدر أغيرها بأي وقت نعم تقدر تغير الباقه من التطبيق أو من خلال خدمة العملاء شكرا لك هل في شيء ثاني؟ لا شكرا حياك الله",
    "T3": "السلام عليكم مركز خدمة العمل كيف أصطيئ مساعتك وعليكم السلام الفاتورة هذا الشهر جاتني أعلى من المعتاد ممكن توضحلي ليش طيب تفضل رقم الجوال المسجل صفر خمسة اتحقق الحين الفاتورة الحالية مية وستة وثمانين ريال والشهر الماضي مية وتسعة وعشرين والزياده بسبب استهلاك بيانات اضافيه تجاوزت الحد في باقتك طيب ما جاني تنبيه تم ارسال اشعار برسالة نصيه عند وصول الاستهلاك ثمانين بالمية شكرا لك فهمت",
    "T4": "السلام عليكم أهلا بك في خدمة عملاء الاتصالات كيف اقدر ساعدك؟ ابغى اعرف كم سعر السياره المعروضه في الموقع حقكم عذرا اقدر اساعدك في استفسارات خدمات الاتصالات بس",
    "T5": "الو يخي ان مرا غضبن منكم وزعلان الا الن محليتو مشكلتي في الانترتت ابغي حل بصرعا ارجكم حسنن بنرفع تلب الهين بمشكلتك",
}

if os.path.exists("classified_calls.jsonl"):
    os.remove("classified_calls.jsonl")
for cid, txt in full_tests.items():
    r = clf.predict_and_save(txt, "classified_calls.jsonl", call_id=cid)
    print(cid, f"({len(txt.split())} كلمة) ->", r["intent"], f"({r['intent_confidence']})",
          "| escalation:", r["escalation"], f"(p={r['escalation_probability']})",
          "| low_conf:", r["low_confidence"])

# تحديث حدود المودل في البطاقة
card_path = f"{PKG}/model_card.json"
card = json.load(open(card_path, encoding="utf-8"))
card["limitations"] = [l for l in card["limitations"] if "Designed for full calls" not in l]
card["limitations"] += [
    "out_of_scope calls in the training data are short and end with the agent's fixed refusal sentence; "
    "the model partly relies on that agent reply. A customer-only off-topic utterance may be mislabeled "
    "(and is usually flagged low_confidence).",
    "Escalation recall is the weak point (~0.90 in CV with a tuned threshold, 2 of 20 missed); "
    "clear escalation cases near the threshold can still be missed.",
]
json.dump(card, open(card_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

shutil.copytree(PKG, "/content/drive/MyDrive/CallSense/" + PKG, dirs_exist_ok=True)
shutil.make_archive(PKG, "zip", PKG)
files.download(PKG + ".zip")

T1 (74 كلمة) -> technical (0.249) | escalation: 0 (p=0.336) | low_conf: True
T2 (61 كلمة) -> information (0.451) | escalation: 0 (p=0.109) | low_conf: False
T3 (66 كلمة) -> billing (0.67) | escalation: 0 (p=0.13) | low_conf: False
T4 (28 كلمة) -> out_of_scope (0.595) | escalation: 0 (p=0.099) | low_conf: False
T5 (22 كلمة) -> technical (0.289) | escalation: 0 (p=0.314) | low_conf: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>